# Inspeção de Qualidade de Peças de Fundição com Visão Computacional

**Mini-Projeto Avaliativo — Módulo 2 (Machine Learning e Visão Computacional)**

Este notebook implementa um pipeline que une **Visão Clássica (OpenCV)** e **Aprendizado Profundo (CNN / TensorFlow-Keras)** para inspecionar automaticamente peças de fundição metálica, classificando-as como **OK** ou **Defeituosa**.

**Etapas:**
1. Análise Exploratória (OpenCV): escala de cinza, blur, limiarização, detecção de bordas e morfologia.
2. Classificação Automatizada (CNN): ingestão em lote, Data Augmentation e treinamento.
3. Auditoria: curvas de Loss e Acurácia (Treino vs. Validação).

In [ ]:
# Bibliotecas de manipulação de dados e visualização
import os
import glob
import zipfile
import numpy as np
import matplotlib.pyplot as plt

# Visão clássica
import cv2

# Deep Learning
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

# Métricas (bônus)
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)

## Sprint 1 — Configuração do Ambiente e Dataset

O dataset é público (*Casting Product Image Data for Quality Inspection*) e **não deve ser versionado no Git** (≈30 MB). Ele fica no Google Drive e é extraído no ambiente do Colab.

In [ ]:
# Monta o Google Drive (somente no Google Colab)
from google.colab import drive
drive.mount('/content/drive')

# Extrai o dataset (zip) do Drive para o ambiente local do Colab
caminho_zip = '/content/drive/MyDrive/casting_line_512x512.zip'
destino = '/content/dataset_pecas'

with zipfile.ZipFile(caminho_zip, 'r') as arquivo_zip:
    arquivo_zip.extractall(destino)

print("Dataset extraído em:", destino)

In [ ]:
# O image_dataset_from_directory espera uma pasta que contenha UMA SUBPASTA POR CLASSE.
# Localizamos automaticamente a pasta que contém as classes 'def_front' e 'ok_front'.
def localizar_pasta_classes(raiz):
    for caminho, subpastas, _ in os.walk(raiz):
        if any(pasta in subpastas for pasta in ('def_front', 'ok_front')):
            return caminho
    raise FileNotFoundError("Não encontrei as pastas de classe 'def_front'/'ok_front'.")

pasta_raiz = localizar_pasta_classes(destino)
print("Pasta de classes:", pasta_raiz)
print("Subpastas:", os.listdir(pasta_raiz))

## Sprint 2 — Análise Exploratória Clássica (OpenCV)

Objetivo: provar visualmente que o defeito (trinca/ranhura) é destacável com processamento clássico.

**Importante:** esta análise é didática e **NÃO é aplicada ao dataset de treino** — servimos para entender o defeito antes de treinar a IA.

In [ ]:
# Seleciona uma amostra de 10 imagens (5 OK e 5 defeituosas)
imagens_ok = sorted(glob.glob(os.path.join(pasta_raiz, 'ok_front', '*')))[:5]
imagens_def = sorted(glob.glob(os.path.join(pasta_raiz, 'def_front', '*')))[:5]
amostra = [('OK', p) for p in imagens_ok] + [('Defeituosa', p) for p in imagens_def]

def carregar_imagem(caminho):
    """Lê a imagem com OpenCV (BGR) e converte para RGB."""
    imagem_bgr = cv2.imread(caminho)
    return cv2.cvtColor(imagem_bgr, cv2.COLOR_BGR2RGB)

print(f"Amostra: {len(imagens_ok)} imagens OK e {len(imagens_def)} defeituosas.")

In [ ]:
# Pipeline clássico — Etapa 1: escala de cinza e suavização de ruído
def pipeline_filtros_basicos(caminho):
    imagem_rgb = carregar_imagem(caminho)
    cinza = cv2.cvtColor(imagem_rgb, cv2.COLOR_RGB2GRAY)
    # Gaussian Blur: média ponderada dos vizinhos (kernel 5x5, sempre ímpar)
    desfoque = cv2.GaussianBlur(cinza, (5, 5), 0)
    return imagem_rgb, cinza, desfoque

# Exibe a amostra completa: Original x Cinza x Blur
fig, eixos = plt.subplots(len(amostra), 3, figsize=(10, 3 * len(amostra)))
for linha, (rotulo, caminho) in enumerate(amostra):
    imagem_rgb, cinza, desfoque = pipeline_filtros_basicos(caminho)
    for eixo, imagem, titulo in zip(
        eixos[linha], [imagem_rgb, cinza, desfoque],
        [f'{rotulo} - Original', 'Escala de cinza', 'Blur (Gaussiano)']
    ):
        eixo.imshow(imagem, cmap='gray')
        eixo.set_title(titulo)
        eixo.axis('off')
plt.tight_layout()
plt.show()

## Sprint 3 — Destaque de Características (Bordas e Morfologia)

Aplicamos limiarização, detecção de bordas (**Canny**) e operações morfológicas (fechamento, erosão e dilatação) para isolar visualmente o defeito da peça.

In [ ]:
# Pipeline clássico completo — Etapa 2: limiarização, bordas e morfologia
def pipeline_completo(caminho):
    imagem_rgb = carregar_imagem(caminho)
    cinza = cv2.cvtColor(imagem_rgb, cv2.COLOR_RGB2GRAY)
    desfoque = cv2.GaussianBlur(cinza, (5, 5), 0)

    # Limiarização: separa objeto (claro) do fundo (escuro)
    _, binaria = cv2.threshold(desfoque, 127, 255, cv2.THRESH_BINARY)

    # Detecção de bordas de Canny (limiar inferior e superior)
    bordas = cv2.Canny(desfoque, 50, 150)

    # Fechamento morfológico: une pequenas descontinuidades das bordas
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    morfologia = cv2.morphologyEx(bordas, cv2.MORPH_CLOSE, kernel)
    return imagem_rgb, cinza, desfoque, binaria, bordas, morfologia

# Plota o pipeline completo para a primeira peça defeituosa
rotulo, caminho = amostra[-1]
imagem_rgb, cinza, desfoque, binaria, bordas, morfologia = pipeline_completo(caminho)

fig, eixos = plt.subplots(1, 5, figsize=(20, 4))
for eixo, imagem, titulo in zip(
    eixos, [imagem_rgb, cinza, desfoque, bordas, morfologia],
    ['Original', 'Escala de cinza', 'Blur', 'Bordas (Canny)', 'Morfologia']
):
    eixo.imshow(imagem, cmap='gray')
    eixo.set_title(titulo)
    eixo.axis('off')
plt.suptitle(f'Pipeline clássico — Peça {rotulo}')
plt.tight_layout()
plt.show()

In [ ]:
# Comparativo: erosão x dilatação x fechamento (kernel 5x5)
kernel5 = np.ones((5, 5), np.uint8)
erosao = cv2.erode(binaria, kernel5, iterations=1)
dilatacao = cv2.dilate(binaria, kernel5, iterations=1)
fechamento = cv2.morphologyEx(binaria, cv2.MORPH_CLOSE, kernel5, iterations=2)

fig, eixos = plt.subplots(1, 4, figsize=(18, 4))
for eixo, imagem, titulo in zip(
    eixos, [binaria, erosao, dilatacao, fechamento],
    ['Binarizada', 'Erosão', 'Dilatação', 'Fechamento']
):
    eixo.imshow(imagem, cmap='gray')
    eixo.set_title(titulo)
    eixo.axis('off')
plt.tight_layout()
plt.show()

### Conclusões da Análise Exploratória

- O **Blur** reduz o ruído industrial (reflexos e granulado do metal), facilitando a detecção.
- O **Canny** evidencia as descontinuidades: a trinca/ranhura aparece como linhas de borda, enquanto a peça OK apresenta contorno mais regular.
- A **morfologia** limpa pequenos ruídos e conecta bordas rompidas.
- Conclusão-chave: *se conseguimos ver o defeito na imagem, a IA também consegue* — por isso a EDA justifica o uso da CNN.

## Sprint 4 — Ingestão de Dados (Keras)

Carregamos o lote completo com `image_dataset_from_directory`, dividindo em **Treino (80%)** e **Validação (20%)**. O dataset já está rotulado pelas subpastas (`ok_front` / `def_front`).

In [ ]:
altura_img, largura_img = 224, 224
tamanho_lote = 32
semente = 123

dados_treino = tf.keras.utils.image_dataset_from_directory(
    pasta_raiz,
    validation_split=0.2,
    subset='training',
    seed=semente,
    image_size=(altura_img, largura_img),
    batch_size=tamanho_lote,
    label_mode='binary'
)

dados_validacao = tf.keras.utils.image_dataset_from_directory(
    pasta_raiz,
    validation_split=0.2,
    subset='validation',
    seed=semente,
    image_size=(altura_img, largura_img),
    batch_size=tamanho_lote,
    label_mode='binary'
)

nomes_classes = dados_treino.class_names
print("Classes:", nomes_classes)

In [ ]:
# Contagem de imagens por classe (balanceamento)
contagens = [len(glob.glob(os.path.join(pasta_raiz, c, '*'))) for c in nomes_classes]

plt.bar(nomes_classes, contagens, color=['steelblue', 'salmon'])
plt.title('Balanceamento de classes')
plt.xlabel('Classe')
plt.ylabel('Quantidade de imagens')
plt.show()

In [ ]:
# Grid de amostras rotuladas
plt.figure(figsize=(12, 6))
for imagens, rotulos in dados_treino.take(1):
    for i in range(10):
        plt.subplot(2, 5, i + 1)
        plt.imshow(imagens[i].numpy().astype('uint8'))
        plt.title(nomes_classes[int(rotulos[i])])
        plt.axis('off')
plt.tight_layout()
plt.show()

## Sprint 4 (cont.) — Data Augmentation

Criamos variações **geométricas** (espelhamento, rotação, zoom) e **luminosas** (brilho, contraste) para simular as variações da esteira industrial e reduzir o overfitting. As camadas só atuam durante o treino.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
], name='data_augmentation')

## Sprint 5 — Arquitetura CNN e Treinamento

CNN Sequencial com blocos **Conv2D + MaxPooling2D** (32 → 64 → 128 filtros, pois as peças têm muitos detalhes), `Flatten`, camada densa, `Dropout` e saída **sigmoide** (classificação binária).

In [ ]:
modelo = models.Sequential([
    layers.Input(shape=(altura_img, largura_img, 3)),
    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
], name='inspetor_fundicao')

modelo.summary()

In [ ]:
modelo.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Freio emergencial: para o treino quando a validação deixa de melhorar
parada_antecipada = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

historico = modelo.fit(
    dados_treino,
    validation_data=dados_validacao,
    epochs=20,
    callbacks=[parada_antecipada],
    verbose=1
)

## Sprint 6 — Auditoria e Análise dos Gráficos

Geramos as curvas de **Acurácia** e **Loss** (Treino vs. Validação) para diagnosticar overfitting.

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(historico.history['accuracy'], label='Treino')
plt.plot(historico.history['val_accuracy'], label='Validação')
plt.title('Acurácia por Época')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(historico.history['loss'], label='Treino')
plt.plot(historico.history['val_loss'], label='Validação')
plt.title('Função de Perda por Época')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Bônus: avaliação detalhada no conjunto de validação
y_verdadeiro = np.concatenate([y for x, y in dados_validacao], axis=0)
probabilidades = modelo.predict(dados_validacao)
y_previsto = (probabilidades > 0.5).astype(int).flatten()

print(classification_report(y_verdadeiro, y_previsto, target_names=nomes_classes))

matriz = confusion_matrix(y_verdadeiro, y_previsto)
plt.figure(figsize=(5, 4))
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues',
            xticklabels=nomes_classes, yticklabels=nomes_classes)
plt.title('Matriz de Confusão')
plt.xlabel('Previsto')
plt.ylabel('Real')
plt.show()

### Diagnóstico: Overfitting ou Aprendizado Saudável?

- **Overfitting ("boca do jacaré"):** a curva de loss de treino continua caindo, enquanto a `val_loss` estagna ou sobe.
- **Aprendizado saudável:** as curvas de treino e validação andam lado a lado, ambas caindo e se estabilizando.
- **Underfitting:** as duas curvas ficam altas/estagnadas.

A acurácia **nunca deve ser analisada sozinha**: um modelo pode ter 99% de acurácia apenas chutando a classe majoritária. Por isso analisamos também o loss e, no bônus, a matriz de confusão (recall da classe "defeituosa").

Para combater o overfitting usamos as técnicas recomendadas em aula: **Data Augmentation**, **Dropout(0.5)**, redução de neurônios densos e mais filtros convolucionais.

## Conclusão

O pipeline une **Visão Clássica** (para entender e evidenciar o defeito) e **CNN** (para classificar em larga escala), simulando a inspeção visual automatizada da Indústria 4.0. A CNN não "sabe" o que é a peça: ela aprende padrões de pixels que separam peças OK das defeituosas.

**Entregáveis:** notebook + `README.md` + vídeo técnico (≤5 min).